# P08 — Prédiction de la conformité sanitaire de l'eau du robinet

**Contexte métier.** ARS (agences régionales de santé), distributeurs d'eau (UDI), particuliers : anticiper les non-conformités bactériologiques ou chimiques d'un prélèvement à partir du contexte (département, type d'installation, mois) pour cibler les contrôles et la modernisation des réseaux. Enjeu : conformité réglementaire, confiance dans l'eau potable.

**Objectif de l'EDA.** Identifier la cible binaire (conformité bactériologique) et les régresseurs discriminants (département, type d'eau, type d'installation, saison).

**Données.** Résultats du contrôle sanitaire de l'eau du robinet — Ministère de la Santé via data.gouv.fr. Licence ouverte Etalab 2.0. Fichier mensuel eaurob (UDI_PLV : prélèvements) : 26 481 prélèvements de janvier 2018 à août 2026.

## 1. Téléchargement

Archive mensuelle des résultats du contrôle sanitaire :

In [ ]:
import urllib.request, zipfile, os
os.makedirs("data", exist_ok=True)

URL = "https://static.data.gouv.fr/resources/resultats-du-controle-sanitaire-de-leau-du-robinet/20261005-113752/eaurob-202608.zip"
if not os.path.exists("data/UDI_PLV_202608.txt"):
    urllib.request.urlretrieve(URL, "data/eaurob-202608.zip")
    with zipfile.ZipFile("data/eaurob-202608.zip") as z:
        for n in z.namelist():
            if "UDI_PLV" in n:
                z.extract(n, "data")
print(os.path.getsize("data/UDI_PLV_202608.txt") / 1e6, "Mo")

## 2. Chargement

Le fichier `UDI_PLV` (une ligne = un prélèvement d'eau) porte la cible : `plvconformitebacterio` et `plvconformitechimique` (O/N).

In [ ]:
import pandas as pd, numpy as np

keep = ["cddept", "inseecommune", "nomcommune", "dateprel", "libtypeeau",
        "conclusionprel", "plvconformitebacterio", "plvconformitechimique",
        "nomreseau", "codetypeinstallation", "nomtypeinstallation", "finaliteprel"]
df = pd.read_csv("data/UDI_PLV_202608.txt", sep=",", usecols=keep, low_memory=False, encoding="latin-1")
df.columns = [c.strip() for c in df.columns]
df["dateprel"] = pd.to_datetime(df["dateprel"], errors="coerce")
df["mois"] = df["dateprel"].dt.month
print(len(df), "prélèvements,", df["inseecommune"].nunique(), "communes")

## 3. Analyse

Taux de non-conformité globale, par département, par mois (saisonnalité des contaminations), par type d'eau et par type d'installation.

In [ ]:
print("Bactériologie :", df["plvconformitebacterio"].value_counts(dropna=False).to_dict())
print("Chimie :", df["plvconformitechimique"].value_counts(dropna=False).to_dict())
ncb = (df["plvconformitebacterio"] == "N").mean() * 100
ncc = (df["plvconformitechimique"] == "N").mean() * 100
print("Non-conformité bactériologique :", round(ncb, 2), "% ; chimique :", round(ncc, 2), "%")
print()
print("NC bactério par mois :", (df.groupby("mois")["plvconformitebacterio"].apply(lambda s: (s == "N").mean() * 100)).round(2).to_dict())
print()
print("NC bactério par type d'eau :", (df.groupby("libtypeeau")["plvconformitebacterio"].apply(lambda s: (s == "N").mean() * 100)).round(2).to_dict())
print("NC bactério par type d'installation :")
print((df.groupby("nomtypeinstallation")["plvconformitebacterio"].apply(lambda s: (s == "N").mean() * 100)).sort_values(ascending=False).head(6).round(2).to_dict())

## 4. Résultats observés (exécution du 16/10/2026)

- n prelevements : 26481
- periode : 2018-01-16 00:00:00 → 2026-08-31 00:00:00
- conclusionprel dist : Eau d'alimentation conforme aux exigences de qualité en vigueur pour l'ensemble des paramètres mesurés. : 13847, Eau …
- bacterio dist : C : 21617, S : 4178, N : 683, nan : 3
- chimique dist : C : 25546, N : 867, D : 56, S : 9, nan : 3
- pct non conforme bacterio : 2.58
- pct non conforme chimique : 3.27
- non conformite bacterio par mois : 1 : 5.06, 2 : 0.75, 3 : 1.11, 4 : 1.45, 5 : 0.87, 6 : 0.75, 7 : 2.24, 8 : 3.00, 9 : 0.00, 10 : 0.00
- non conformite par type eau : EAU BRUTE SOUTERRAINE  : 0.00, EAU DISTRIBUEE DESINFECTEE : 1.68, EAU DISTRIBUEE SANS DESINFECTION : 12.67, EAU SUPER…
- non conformite par type installation : UNITE DE DISTRIBUTION : 2.58
- n communes : 13206
- n reseaux : 13735

## 5. Variable cible et régresseurs

**Cible (classification binaire) : `plvconformitebacterio` = N (prélèvement non conforme bactériologiquement).** Classe très déséquilibrée (~1 %) : churn-like. Variante : non-conformité chimique.

**Régresseurs retenus :**
- `mois` : saisonnalité des contaminations (été plus critique)
- `cddept` / `inseecommune` : géographie des réseaux
- `libtypeeau` (eau distribution / forage / source) et `nomtypeinstallation` (UDI, puits, réseau…) : certains types d'installations concentrent les NC
- `finaliteprel` (motif du contrôle : routine, plainte, suivi de non-conformité — attention ce dernier est un biais)

**Pièges :**
- La conformité chimique n'est mesurée que sur sous-échantillonnage fréquentiel : ne pas mélanger les deux cibles
- `finaliteprel` contient des contrôles déclenchés par une non-conformité passée : fuite potentielle — l'analyser avant de l'inclure
- Encodage latin-1 du fichier
- Le taux de NC brut (~1,2 % bactériologique) rend l'accuracy trompeuse : évaluer en recall/precision par classe ou en AUC